<a href="https://colab.research.google.com/github/samjurassic/datascience-demo/blob/main/coda/HBS_CoDA_Python_Part2_NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HBS CoDA: NLP With Python

Learning objectives include:

• Understand vector embeddings of text, TF-IDF matrices

• Build a topic model on a corpus of text using BERTopic and UMAP

• Visualize topic model results

• Use a local LLM for classifying text data

In [ ]:
import math
import warnings
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import matplotlib.pyplot as plt
import seaborn as sns

## Constructing the TF-iDF Matrix

In [ ]:
# A tiny, low-dimensional dataset
corpus = [
    "the robot makes a car",     # Doc 0
    "the robot makes a pizza",   # Doc 1
    "the human eats a pizza"     # Doc 2
]

# Extract the vocabulary (all unique words, sorted alphabetically)
vocab = sorted(set(word for doc in corpus for word in doc.split()))
print(f"Vocabulary ({len(vocab)} words): {vocab}\n")

# Calculate TF (Term Frequency)
# Formula: (Count of word in doc) / (Total words in doc)
def compute_tf(corpus, vocab):
    tf_data = []
    for doc in corpus:
        words = doc.split()
        doc_length = len(words)

        # Count words and divide by total words in that document
        tf_row = {word: words.count(word) / doc_length for word in vocab}
        tf_data.append(tf_row)

    return pd.DataFrame(tf_data, index=["Doc 0", "Doc 1", "Doc 2"])

tf_matrix = compute_tf(corpus, vocab)
print("--- TERM FREQUENCY (TF) MATRIX ---")
display(tf_matrix) # 'display' looks really nice in Colab

In [ ]:
# Calculate IDF (Inverse Document Frequency)
# Formula: log(Total Documents / Number of Documents containing the word)
# IDF = 0 if in every document (100% = 1.0 = 10^0)

def compute_idf(corpus, vocab):
    N = len(corpus) # Total documents (3)
    idf_dict = {}

    for word in vocab:
        # How many documents contain this word?
        doc_count = sum(1 for doc in corpus if word in doc.split())

        # Calculate log (using base 10 for easier human reading)
        idf_dict[word] = math.log10(N / doc_count)

    return pd.DataFrame([idf_dict], index=["IDF Score"])

idf_matrix = compute_idf(corpus, vocab)
print("\n--- INVERSE DOCUMENT FREQUENCY (IDF) SCORES ---")
display(idf_matrix)

In [ ]:
# Calculate final TF-IDF Matrix
# Formula: TF * IDF
def compute_tfidf(tf_matrix, idf_matrix):
    # Multiply the TF dataframe by the IDF dataframe row by row
    tfidf_matrix = tf_matrix.multiply(idf_matrix.iloc[0], axis=1)
    return tfidf_matrix

final_tfidf = compute_tfidf(tf_matrix, idf_matrix)
print("\n--- FINAL TF-IDF MATRIX ---")
display(final_tfidf)

In [ ]:
# Note: Sklearn uses slightly more complex math behind the scenes
# (L2 normalization and smoothed IDF) to prevent division-by-zero errors,
# but the core concept is identical!

vectorizer = TfidfVectorizer()
sklearn_matrix = vectorizer.fit_transform(corpus)

# Convert to a dataframe so we can see it
sklearn_df = pd.DataFrame(
    sklearn_matrix.toarray(),
    columns=vectorizer.get_feature_names_out(),
    index=["Doc 0", "Doc 1", "Doc 2"]
)

print("\n--- SCIKIT-LEARN TF-IDF MATRIX ---")
display(sklearn_df)

In [ ]:
# 1. Scikit-Learn's TF is just the raw count of words
tf_raw = []
for doc in corpus:
    words = doc.split()
    tf_raw.append([words.count(word) for word in vocab])
tf_raw = np.array(tf_raw)

# 2. Scikit-Learn's smoothed IDF formula: ln((1+N) / (1+df)) + 1
N = len(corpus)
idf_sklearn = []
for word in vocab:
    df = sum(1 for doc in corpus if word in doc.split())
    # Note the use of np.log (natural log) and the +1 smoothing
    idf_score = np.log((1 + N) / (1 + df)) + 1
    idf_sklearn.append(idf_score)
idf_sklearn = np.array(idf_sklearn)

# 3. Multiply TF * IDF
tfidf_raw = tf_raw * idf_sklearn

# 4. L2 Normalization (Scale each row so its Euclidean length is 1)
# Formula for a row: row / sqrt(sum(row^2))
row_norms = np.sqrt(np.sum(tfidf_raw**2, axis=1, keepdims=True))
tfidf_l2_normalized = tfidf_raw / row_norms

# Let's compare!
print("--- OUR MANUAL 'SKLEARN-STYLE' MATRIX ---")
display(pd.DataFrame(tfidf_l2_normalized, columns=vocab, index=["Doc 0", "Doc 1", "Doc 2"]).round(4))

print("\n--- ACTUAL SCIKIT-LEARN MATRIX ---")
display(sklearn_df.round(4))

In [ ]:
# By default, sklearn ignores single-character words.
# We can override the 'token_pattern' to include words of length 1 (\w+)
vectorizer = TfidfVectorizer(token_pattern=r'(?u)\b\w+\b')

sklearn_matrix = vectorizer.fit_transform(corpus)

# Convert to a dataframe so we can see it
sklearn_df = pd.DataFrame(
    sklearn_matrix.toarray(),
    columns=vectorizer.get_feature_names_out(),
    index=["Doc 0", "Doc 1", "Doc 2"]
)

print("\n--- SCIKIT-LEARN TF-IDF MATRIX (With 'a' included) ---")
display(sklearn_df.round(4))
print("--- OUR MANUAL 'SKLEARN-STYLE' MATRIX ---")
display(pd.DataFrame(tfidf_l2_normalized, columns=vocab, index=["Doc 0", "Doc 1", "Doc 2"]).round(4))

## Cosine Similarity

In [ ]:
# Let's create 3 imaginary documents reduced to just 2 dimensions (X and Y)
# Think of X as "Pet-ness" and Y as "Finance-ness"
vec_A = np.array([[0.8, 0.1]])  # "I love my dog"
vec_B = np.array([[0.9, 0.2]])  # "Puppies are great" (Similar angle to A)
vec_C = np.array([[0.1, 0.9]])  # "Stock market crashed" (Completely different angle)

# Calculate Cosine Similarities against Vector A
sim_AB = cosine_similarity(vec_A, vec_B)[0][0]
sim_AC = cosine_similarity(vec_A, vec_C)[0][0]

print(f"Similarity A to B: {sim_AB:.4f}")
print(f"Similarity A to C: {sim_AC:.4f}")

In [ ]:
# --- PLOTTING ---
plt.figure(figsize=(8, 6))
ax = plt.gca()

# Plot the vectors as arrows from the origin (0,0)
ax.quiver(0, 0, vec_A[0,0], vec_A[0,1], angles='xy', scale_units='xy', scale=1, color='blue', label=f'Doc A (Dog)')
ax.quiver(0, 0, vec_B[0,0], vec_B[0,1], angles='xy', scale_units='xy', scale=1, color='cyan', label=f'Doc B (Puppy) | Sim to A: {sim_AB:.3f}')
ax.quiver(0, 0, vec_C[0,0], vec_C[0,1], angles='xy', scale_units='xy', scale=1, color='red',  label=f'Doc C (Stocks) | Sim to A: {sim_AC:.3f}')

# Formatting the plot to look good in Colab
plt.xlim(0, 1)
plt.ylim(0, 1)
plt.grid(True, linestyle='--', alpha=0.6)
plt.title("Cosine Similarity: Measuring the Angle Between Vectors", fontsize=14)
plt.xlabel("Dimension 1 (e.g., Pet-ness)")
plt.ylabel("Dimension 2 (e.g., Finance-ness)")
plt.legend(loc='upper left')
plt.show()

## Embeddings

Embeddings are lower-dimensional representations of a corpus. They are typically created by neural networks that use a smaller hidden layer as a means of reducing the input vector's size.

[More on embeddings](https://developers.google.com/machine-learning/crash-course/embeddings/embedding-space)

In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
# 1. Three sentences: Two are semantically similar (but share no nouns!), one is different.
sentences = [
    "The golden retriever chased the tennis ball.",
    "A happy dog ran after the round toy.",
    "Interest rates were raised by the federal reserve today."
]

# 2. Get the real 384-dimensional dense vectors
dense_vectors = embedding_model.encode(sentences)
print(f"Embeddings Shape: {dense_vectors.shape}")

# 3. Calculate the Cosine Similarity Matrix
# This compares every sentence against every other sentence
similarity_matrix = cosine_similarity(dense_vectors)

# 4. Plotting the Heatmap
plt.figure(figsize=(7, 5))
sns.heatmap(
    similarity_matrix,
    annot=True,          # Show the exact numbers
    cmap='Blues',        # Dark blue = highly similar
    vmin=0, vmax=1,      # Cosine sim for text is usually between 0 and 1
    xticklabels=["Doc 0 (Retriever)", "Doc 1 (Dog)", "Doc 2 (Finance)"],
    yticklabels=["Doc 0", "Doc 1", "Doc 2"]
)

plt.title("Real 384-D Text Embeddings Similarity Matrix", fontsize=14)
plt.xticks(rotation=15)
plt.show()

## BERTopic

"BERTopic is a topic modeling technique that leverages 🤗 transformers and c-TF-IDF to create dense clusters allowing for easily interpretable topics whilst keeping important words in the topic descriptions."

1. Embed text
2. Reduce dimensions (UMAP).
3. Cluster (HDBSCAN).
4. Extract topic words (c-TF-IDF).

[BERTopic Docs](https://maartengr.github.io/BERTopic/index.html)

[UMAP Docs](https://umap-learn.readthedocs.io/en/latest/)

[HDBSCAN Docs](https://scikit-learn.org/stable/modules/clustering.html#hdbscan)

In [ ]:
# we will install bertopic using pip because it doesn't come with Colab.
# Only need to run once per session.

!pip install --quiet bertopic datasets

from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN

In [ ]:
# A small toy dataset to demonstrate clustering
docs = [
    # Topic 1: Baseball
    "The pitcher looked in for the sign before winding up for the throw.",
    "A loud crack echoed through the stadium as the bat connected with the ball.",
    "The umpire dusted off home plate and yelled for the batter to step in.",
    "Fans scrambled to catch the foul ball that landed in the upper deck.",
    "The game went into extra innings after the score remained tied in the ninth.",
    "He slid into second base just barely beating the tag from the shortstop.",
    "Managing the bullpen is a critical strategy for winning close games.",
    "The outfielder made a spectacular diving catch to save a run.",
    "Spring training allows players to warm up before the regular season begins.",
    "Hitting a grand slam is one of the most exciting moments in the sport.",

    # Topic 2: Steely Dan
    "Walter Becker and Donald Fagen formed the core of the band known for complex jazz structures.",
    "The album Aja is often cited as a masterpiece of audio engineering and production.",
    "Their lyrics often feature sarcastic humor and cryptic storytelling about eccentric characters.",
    "Steely Dan is famous for using a revolving door of top-tier session musicians.",
    "The blend of jazz harmonies with rock rhythms creates their signature sophisticated sound.",
    "Reelin' In the Years features one of the most recognizable guitar solos in classic rock.",
    "They were known for their obsessive perfectionism in the recording studio.",
    "After a long hiatus, the band returned to touring in the early nineties to great acclaim.",
    "The song Deacon Blues describes the fantasy of becoming a jazz saxophonist.",
    "Their music often defies easy categorization, bridging the gap between pop and jazz fusion.",

    # Topic 3: Seagulls
    "The seagull swooped down and snatched a french fry right out of my hand.",
    "Large flocks of white birds gathered along the shoreline during low tide.",
    "Their piercing cries can be heard echoing across the boardwalk early in the morning.",
    "Seagulls are incredibly opportunistic feeders and will eat almost anything they find.",
    "The grey and white feathers help them blend in with the cloudy coastal skies.",
    "Tourists are often warned not to feed the birds to avoid aggressive behavior.",
    "A solitary gull perched on the wooden piling, watching the fishing boats return.",
    "These coastal birds have adapted well to living in urban environments near the sea.",
    "They build their nests on high cliffs to protect their eggs from predators.",
    "The span of their wings allows them to glide effortlessly over the ocean currents.",

    # Topic 4: Documentaries
    "Documentary films aim to capture reality through a non-fictional narrative lens.",
    "Cinéma vérité is a style of filmmaking that emphasizes observational techniques and raw footage.",
    "The ethics of a documentary often revolve around the director's responsibility to the subject's truth.",
    "Wildlife documentaries use high-definition cameras to observe animal behavior in their natural habitats.",
    "Historical documentaries often rely on archival footage and interviews with experts to reconstruct the past.",
    "Social justice films use the power of the documentary format to highlight systemic inequalities.",
    "A compelling voiceover can guide the viewer through complex information in an educational film.",
    "In the editing room, hundreds of hours of raw documentary footage must be distilled into a coherent story.",
    "While mockumentaries use the same visual language, they are actually scripted satires rather than real documentaries.",
    "The recent surge in true crime documentaries has changed how audiences engage with legal and criminal justice stories.",
]

df = pd.DataFrame({"text": docs})

# Look at the text
print(df['text'].head())
docs = df['text'].tolist()

# Generate embeddings for the FULL dataset (docs)
# We name the variable 'embeddings' so it matches our BERTopic code
embeddings = embedding_model.encode(docs, show_progress_bar=True)

print(f"Generated {len(embeddings)} vectors, each with {embeddings.shape[1]} dimensions.")

# --- STEP 1: Setup Models (from previous step) ---
# We use small-data settings so it works on the 30 sentences
umap_model = UMAP(
    n_neighbors=3,
    n_components=3,
    min_dist=0.0,
    metric='cosine',
    random_state=1234)

hdbscan_model = HDBSCAN(
    min_cluster_size=3,
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True)

# --- STEP 2: Configure Stopword Removal ---
vectorizer_model = CountVectorizer(stop_words="english")

# --- STEP 3: Run BERTopic ---
topic_model = BERTopic(
    embedding_model=embedding_model,   # <--- 1. Tell it WHICH model you used
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    language="english"
)

# <--- 2. Pass the PRE-CALCULATED embeddings into fit_transform!
topics, probs = topic_model.fit_transform(docs, embeddings=embeddings)

print(f"Discovered {len(topic_model.get_topic_info())} topics!")



In [ ]:
# This matrix contains the scores for every word in every topic.
# Higher score = This word is more unique/important to this topic.

# Get the matrix (Topics x Words)
c_tf_idf_matrix = topic_model.c_tf_idf_

# Get the list of feature names (words)
feature_names = topic_model.vectorizer_model.get_feature_names_out()

# Convert to a Pandas DataFrame for easy viewing
df_tfidf = pd.DataFrame(
    c_tf_idf_matrix.toarray(),
    index=[f"Topic {i}" for i in topic_model.topic_labels_.keys()],
    columns=feature_names
)

# Show a subset of interesting words to prove it worked
# We pick words we know should be in our 3 topics
interesting_words = ["baseball", "pitcher", "guitar", "jazz", "bird", "seagull", "film", "footage"]

# Filter the dataframe to show only these words (if they exist in the vocab)
# Note: We use an intersection check in case a word was dropped
valid_cols = [w for w in interesting_words if w in df_tfidf.columns]
print(df_tfidf[valid_cols].round(3))

In [ ]:
# Visualize the actual documents (dots) in vector space
# heavily reduces dimensionality to 2D so we can see it
fig = topic_model.visualize_documents(docs)

# This returns a Plotly figure (interactive)
fig.show()

In [ ]:
topic_model.visualize_topics()

In [ ]:
# Visualize the keywords in each topic and their c-TF-IDF scores
topic_model.visualize_barchart()

In [ ]:
# see information about topic number 2
topic_model.get_topic_info(2).T

In [ ]:
topic_numbers = topic_model.get_topics().keys()

# This creates a comprehensive dataframe for all documents
doc_info = topic_model.get_document_info(docs)

for tn in topic_numbers:
  display(doc_info.query("Topic == @tn"))
  print("\n")


## Training an LLM for Classification

In [ ]:
from datasets import load_dataset
from transformers import (
    pipeline,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

# Load the dataset - this is data from banking customer support
dataset = load_dataset("banking77")

# We map the official IDs to our local 0, 1, 2, 3 indices
label_map = {
    41: 0, # Official 'lost_or_stolen_card'
    17: 1, # Official 'card_payment_wrong_exchange_rate'
    25: 2, # Official 'declined_card_payment'
    13: 3  # Official 'card_linking'
}

# These must be in the exact order of the numbers above (0, 1, 2, 3)
category_names = ["Lost/Stolen Card", "Foreign Transaction Fees", "Declined Card Payment", "Card Linking"]

# FILTER & MAP
def prepare_data(example):
    example['label'] = label_map[example['label']]
    return example

# Filter first, then map the labels
train_data = dataset['train'].filter(lambda x: x['label'] in label_map.keys())
train_dataset = train_data.map(prepare_data).shuffle(seed=42).select(range(400))

test_data = dataset['test'].filter(lambda x: x['label'] in label_map.keys())
test_dataset = test_data.map(prepare_data).shuffle(seed=42).select(range(100))

id2label = {i: name for i, name in enumerate(category_names)}
label2id = {name: i for i, name in enumerate(category_names)}


model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=4,
    id2label=id2label,
    label2id=label2id
)

In [ ]:
# check labels
print(train_dataset.to_pandas()[['text', 'label']].query("label == 1").head())

In [ ]:
# Load the Tokenizer (https://huggingface.co/docs/transformers/en/model_doc/distilbert)
model_id = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_id)

def tokenize_function(examples):
    # Padding and truncation ensure all sentences are the same mathematical length
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=64)

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

# Create the dictionaries for the model config
id2label = {i: name for i, name in enumerate(category_names)}
label2id = {name: i for i, name in enumerate(category_names)}

print(f"Current Mapping: {id2label}")

print("Model and Tokenizer ready!")

In [ ]:
# Mute warnings for a clean output
warnings.filterwarnings("ignore")

# Define the training rules
training_args = TrainingArguments(
    output_dir="./banking_model",
    learning_rate=2e-5, # affects how quickly the model updates weights
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=10,      # Run through the data N times
    weight_decay=0.01,
    fp16=True,               # Supercharges speed on the Colab T4 GPU!
    eval_strategy="epoch",
    logging_steps=10
)

# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    processing_class=tokenizer,
)

print("Starting Fine-Tuning...")
trainer.train()
print("Training Complete!")

In [ ]:
# Create the pipeline using the TRAINED model object
# Important: ensure 'model' here is the one you just ran trainer.train() on!
custom_classifier = pipeline("text-classification", model=model, tokenizer=tokenizer, device=0)

sample_tickets = [
    "I lost my wallet on the train this morning, please cancel my Visa!", # Should be 0
    "Why did I get charged a 3% fee for buying a coffee in Paris?",     # Should be 1
    "The mobile app keeps crashing when I try to view my PIN."          # Should be 2
]

results = []
for ticket in sample_tickets:
    pred = custom_classifier(ticket)[0]
    results.append({
        "Text": ticket,
        "Label": pred['label'],      # This should now be the string name
        "Score": f"{pred['score']:.1%}"
    })

display(pd.DataFrame(results))